# PNPL 2026 — submission smoke test

Tiny end-to-end Colab notebook that:
1. installs `pnpl` (from this branch) and the latest `kaggle` CLI,
2. builds a randomly-initialised submission CSV in the required format,
3. uploads it to the PNPL 2026 competition on Kaggle.

The submission format is `index, <50 primary-vocab probs>, <50 moses-vocab probs (prefixed `moses_`)>`. Only the primary distribution is scored (balanced accuracy on argmax); the moses block rides along for downstream use.

**Auth:** the submission step needs a Kaggle API token. Generate one at <https://www.kaggle.com/settings/api> (the modern token looks like `KGAT_…`). Don't paste it into the notebook source — keep it in a Colab secret or the runtime env.

## 1. Install dependencies

In [ ]:
# Pin to >=2.0 for KAGGLE_API_TOKEN (KGAT_…) support; needs Python >= 3.11 (Colab default in 2026).
%pip install -q --upgrade "kaggle>=2.0" numpy pandas
# Install pnpl from GitHub. Replace the URL with a local install if you're hacking on the package.
%pip install -q git+https://github.com/neural-processing-lab/pnpl-public.git

## 2. Configure Kaggle credentials

On Colab, prefer storing the token as a secret. Two common patterns:

**Option A — Colab user secret (recommended).** Click the 🔑 icon in the left sidebar, add a secret named `KAGGLE_API_TOKEN`, enable access for this notebook, then run the cell below.

**Option B — paste at runtime.** Run `getpass.getpass()` and paste; the token never enters the notebook source.

Either way, the token is exported as `KAGGLE_API_TOKEN` for the rest of the session.

In [ ]:
import os

if not os.environ.get("KAGGLE_API_TOKEN"):
    try:
        from google.colab import userdata  # type: ignore
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        import getpass
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Kaggle API token (KGAT_…): ").strip()

assert os.environ["KAGGLE_API_TOKEN"].startswith("KGAT_"), \
    "Expected a modern Kaggle API token starting with 'KGAT_'. Generate one at https://www.kaggle.com/settings/api"
print("token configured (length =", len(os.environ["KAGGLE_API_TOKEN"]), ")")

## 3. Build a submission CSV from the holdout data

`LibriBrainCompetitionHoldout` downloads the holdout recordings from Hugging Face
(`pnpl/LibriBrain-Competition-2026`) and enumerates the **canonical rows** of a
submission — one row per word. Pick a track:

- `track="deep"` → subject 0 (within-subject / Deep track)
- `track="broad"` → subjects 1–39 (cross-subject / Broad track)

Each row yields a `(306, 250)` MEG window (1 s @ 250 Hz). Below we feed those
windows through a **placeholder** model that emits uniform-random probabilities —
replace `predict()` with your model to make a real submission. The `index` column
comes straight from `holdout.indices`, so it aligns with the leaderboard solution
by construction.

In [ ]:
import numpy as np
from pnpl.competition import (
    PRIMARY_VOCAB, SECONDARY_VOCAB,
    LibriBrainCompetitionHoldout, write_submission,
)

print(f"primary vocab ({len(PRIMARY_VOCAB)}): {PRIMARY_VOCAB[:5]} ... {PRIMARY_VOCAB[-3:]}")
print(f"moses vocab   ({len(SECONDARY_VOCAB)}): {SECONDARY_VOCAB[:5]} ... {SECONDARY_VOCAB[-3:]}")

TRACK = "deep"  # "deep" (subject 0) or "broad" (subjects 1-39)
holdout = LibriBrainCompetitionHoldout(track=TRACK)   # downloads from HF on first use
print(f"\n{TRACK} track: {len(holdout)} rows  {holdout.counts()}")


def predict(meg_batch):
    """Placeholder model. meg_batch: (B, 306, 250) -> (primary (B,50), moses (B,50)).

    Replace with your trained model. Here: uniform-random softmax logits.
    """
    b = meg_batch.shape[0]
    g = np.random.default_rng(0)
    def softmax(x):
        e = np.exp(x - x.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)
    return softmax(g.standard_normal((b, 50))), softmax(g.standard_normal((b, 50)))


primary_probs, secondary_probs = [], []
for meg_batch, _metas in holdout.iter_windows(batch_size=256):
    p, m = predict(meg_batch)
    primary_probs.append(p); secondary_probs.append(m)
primary_probs = np.concatenate(primary_probs)
secondary_probs = np.concatenate(secondary_probs)

csv_path = write_submission(
    f"submission_{TRACK}.csv",
    indices=holdout.indices,           # canonical row order — do not reorder
    primary_probs=primary_probs,       # (N, 50) competition vocab, scored
    secondary_probs=secondary_probs,   # (N, 50) Moses-50, secondary metric
)
print(f"wrote {csv_path}")

import pandas as pd
pd.read_csv(csv_path).iloc[:3, :4]

## 4. Submit to Kaggle

In [ ]:
from pnpl.competition import submit_to_kaggle

# Prints a one-line ✅/❌ status. The returned result is truthy on success and
# carries Kaggle's response on `.detail` (plus `.stdout`/`.stderr`/`.process`
# for the raw CLI output). If credentials are missing it raises a clear error
# instead of a traceback.
result = submit_to_kaggle(
    csv_path,
    competition="pnpl-competition-2026",
    message="colab smoke test — uniform random probs",
)
result

## 5. (Optional) Check submission status

After submitting, Kaggle takes a few seconds to score. List your recent submissions to see the leaderboard score.

In [ ]:
!kaggle competitions submissions -c pnpl-competition-2026 | head -10